# OHLCV-Only Market Regime Detection & Backtest
For a **single liquid stock/index**, using only daily Open/High/Low/Close/Volume.

Pipeline:
1. Load data
2. Engineer factors (returns, volatility, trend, range, volume, liquidity, statistical)
3. Fit a Gaussian HMM to detect regimes (unsupervised)
4. Label regimes (low-vol trending / high-vol choppy / stress) by their statistical properties
5. Walk-forward backtest: does knowing yesterday's regime help predict next few days' behavior?

**Requirements:** `pip install pandas numpy scipy matplotlib hmmlearn --break-system-packages` (drop the flag if not on Debian/Ubuntu managed Python)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option('display.width', 120)
plt.rcParams['figure.figsize'] = (12, 5)


## 1. Load your OHLCV data

Expected columns (case-insensitive, will be auto-renamed): `date, open, high, low, close, volume`.

Two options below — use whichever fits.


In [ ]:
# --- OPTION A: load from a CSV you already have ---
# df = pd.read_csv("your_file.csv")

# --- OPTION B: pull from yfinance (uncomment if you have internet + yfinance installed) ---
# import yfinance as yf
# df = yf.download("RELIANCE.NS", start="2018-01-01", end="2026-09-01").reset_index()

# --- OPTION C: quick synthetic sample so the notebook runs standalone right now ---
np.random.seed(42)
n = 1500
dates = pd.bdate_range("2019-01-01", periods=n)
# simulate regime-switching returns so the notebook has something realistic to detect
regimes = np.random.choice([0, 1, 2], size=n, p=[0.6, 0.3, 0.1])
mu_map = {0: 0.0004, 1: -0.0002, 2: -0.0015}
sig_map = {0: 0.008, 1: 0.015, 2: 0.035}
rets = np.array([np.random.normal(mu_map[r], sig_map[r]) for r in regimes])
close = 100 * np.exp(np.cumsum(rets))
high = close * (1 + np.abs(np.random.normal(0, 0.006, n)))
low = close * (1 - np.abs(np.random.normal(0, 0.006, n)))
open_ = low + (high - low) * np.random.rand(n)
volume = np.random.lognormal(mean=14, sigma=0.4, size=n) * (1 + 2 * (regimes == 2))

df = pd.DataFrame({
    "date": dates, "open": open_, "high": high, "low": low,
    "close": close, "volume": volume
})

df.columns = [c.strip().lower() for c in df.columns]
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
df = df.set_index("date")
df.tail()


## 2. Feature engineering

All factors are computed **causally** (only using data up to and including day *t*) so there is no lookahead bias.


In [ ]:
def engineer_features(df, price_col="close"):
    d = df.copy()

    # ---- Returns ----
    d["ret_1d"] = d[price_col].pct_change()
    d["logret_1d"] = np.log(d[price_col] / d[price_col].shift(1))
    for h in [3, 5, 10, 21, 63]:
        d[f"ret_{h}d"] = d[price_col].pct_change(h)

    # ---- Volatility estimators ----
    # Close-to-close rolling realized vol (annualized)
    for w in [5, 10, 20, 60]:
        d[f"vol_cc_{w}"] = d["logret_1d"].rolling(w).std() * np.sqrt(252)

    # Parkinson estimator (uses High-Low range)
    hl_ratio = np.log(d["high"] / d["low"])
    d["parkinson_1d"] = hl_ratio ** 2 / (4 * np.log(2))
    d["parkinson_20"] = np.sqrt(d["parkinson_1d"].rolling(20).mean() * 252)

    # Garman-Klass estimator (uses O,H,L,C)
    log_hl = np.log(d["high"] / d["low"]) ** 2
    log_co = np.log(d["close"] / d["open"]) ** 2
    gk_1d = 0.5 * log_hl - (2 * np.log(2) - 1) * log_co
    d["gk_1d"] = gk_1d
    d["gk_20"] = np.sqrt(d["gk_1d"].rolling(20).mean().clip(lower=0) * 252)

    # Vol-of-vol
    d["vol_of_vol_20"] = d["vol_cc_20"].rolling(20).std()

    # ATR (normalized)
    prev_close = d["close"].shift(1)
    tr = pd.concat([
        d["high"] - d["low"],
        (d["high"] - prev_close).abs(),
        (d["low"] - prev_close).abs()
    ], axis=1).max(axis=1)
    d["atr_14"] = tr.rolling(14).mean()
    d["atr_pct_14"] = d["atr_14"] / d["close"]

    # ---- Trend / technical ----
    d["ma_20"] = d["close"].rolling(20).mean()
    d["ma_50"] = d["close"].rolling(50).mean()
    d["ma_200"] = d["close"].rolling(200).mean()
    d["dist_ma20_pct"] = (d["close"] - d["ma_20"]) / d["ma_20"]
    d["dist_ma50_pct"] = (d["close"] - d["ma_50"]) / d["ma_50"]
    d["ma20_slope"] = d["ma_20"].pct_change(5)

    # Rolling autocorrelation of returns (trend vs mean-reversion signal)
    def rolling_autocorr(x, lag=1):
        return x.rolling(20).apply(lambda s: pd.Series(s).autocorr(lag=lag), raw=False)
    d["autocorr_20"] = rolling_autocorr(d["logret_1d"], lag=1)

    # RSI (14-day, and a fast 3-day version for short-horizon mean reversion)
    def rsi(series, window):
        delta = series.diff()
        gain = delta.clip(lower=0).rolling(window).mean()
        loss = (-delta.clip(upper=0)).rolling(window).mean()
        rs = gain / loss.replace(0, np.nan)
        return 100 - (100 / (1 + rs))
    d["rsi_14"] = rsi(d["close"], 14)
    d["rsi_3"] = rsi(d["close"], 3)

    # ADX-lite: trend strength proxy via normalized directional movement
    up_move = d["high"].diff()
    down_move = -d["low"].diff()
    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)
    atr14 = d["atr_14"].replace(0, np.nan)
    plus_di = 100 * pd.Series(plus_dm, index=d.index).rolling(14).mean() / atr14
    minus_di = 100 * pd.Series(minus_dm, index=d.index).rolling(14).mean() / atr14
    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di)
    d["adx_14"] = dx.rolling(14).mean()

    # ---- Range / price structure ----
    d["hl_range_pct"] = (d["high"] - d["low"]) / d["close"]
    d["gap_pct"] = (d["open"] - d["close"].shift(1)) / d["close"].shift(1)
    d["close_position"] = (d["close"] - d["low"]) / (d["high"] - d["low"]).replace(0, np.nan)
    d["drawdown_20"] = d["close"] / d["close"].rolling(20).max() - 1
    d["drawdown_60"] = d["close"] / d["close"].rolling(60).max() - 1

    # ---- Volume ----
    d["vol_avg_20"] = d["volume"].rolling(20).mean()
    d["rel_volume"] = d["volume"] / d["vol_avg_20"]
    d["dollar_volume"] = d["close"] * d["volume"]
    obv_direction = np.sign(d["close"].diff()).fillna(0)
    d["obv"] = (obv_direction * d["volume"]).cumsum()

    # ---- Liquidity checks ----
    d["amihud"] = (d["ret_1d"].abs() / d["dollar_volume"]).replace([np.inf, -np.inf], np.nan)
    d["zero_ret_flag"] = (d["ret_1d"] == 0).astype(int)

    # ---- Statistical / distribution ----
    d["skew_20"] = d["logret_1d"].rolling(20).skew()
    d["kurt_20"] = d["logret_1d"].rolling(20).kurt()
    d["zscore_ret_60"] = (d["ret_1d"] - d["ret_1d"].rolling(60).mean()) / d["ret_1d"].rolling(60).std()

    def hurst_exponent(ts):
        ts = np.asarray(ts)
        if len(ts) < 20 or np.any(np.isnan(ts)):
            return np.nan
        lags = range(2, 20)
        tau = [np.std(np.subtract(ts[lag:], ts[:-lag])) for lag in lags]
        tau = [t if t > 0 else 1e-8 for t in tau]
        poly = np.polyfit(np.log(list(lags)), np.log(tau), 1)
        return poly[0] * 2.0

    d["hurst_60"] = d["close"].rolling(60).apply(hurst_exponent, raw=False)

    return d

feat = engineer_features(df)
feat.tail()


## 3. Compact, low-collinearity feature set for regime detection

Using a small, complementary basket (volatility, trend/mean-reversion, range, volume, momentum)
rather than everything above — feeding highly correlated features into an HMM tends to just
duplicate one underlying signal and destabilizes the fit.


In [ ]:
regime_features = [
    "vol_cc_20",       # volatility level
    "autocorr_20",     # trend vs mean-reversion
    "dist_ma20_pct",   # trend direction/strength
    "atr_pct_14",      # range regime
    "rel_volume",       # activity/conviction
    "close_position",  # intraday sentiment proxy
    "drawdown_20",      # momentum/stress
]

model_df = feat[regime_features].dropna()
print(model_df.shape)
model_df.describe()


## 4. Fit a Gaussian HMM

Install if needed: `pip install hmmlearn --break-system-packages`

3 states is a common starting point: e.g. *calm/trending*, *choppy/neutral*, *stressed/high-vol*.
Standardize features first since HMM components are sensitive to scale.


In [ ]:
from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler

N_STATES = 3

scaler = StandardScaler()
X = scaler.fit_transform(model_df.values)

hmm = GaussianHMM(n_components=N_STATES, covariance_type="full",
                   n_iter=1000, random_state=42)
hmm.fit(X)

hidden_states = hmm.predict(X)
model_df = model_df.copy()
model_df["state"] = hidden_states
model_df.head()


## 5. Label states by their statistical properties

The HMM gives you arbitrary state indices (0,1,2...) — inspect their mean vol/trend to assign
human-readable labels.


In [ ]:
state_summary = model_df.groupby("state")[regime_features].mean()
state_summary["count"] = model_df.groupby("state").size()
state_summary["pct_time"] = (state_summary["count"] / len(model_df) * 100).round(1)
state_summary = state_summary.sort_values("vol_cc_20")
print(state_summary)

# simple auto-labeling by volatility rank (lowest vol = calm, highest = stressed)
vol_rank = state_summary["vol_cc_20"].rank().astype(int)
label_map = {}
sorted_states = state_summary.index.tolist()
names = ["Calm/Trending", "Choppy/Neutral", "Stressed/High-Vol"][:N_STATES]
for name, st in zip(names, sorted_states):
    label_map[st] = name
print(label_map)

model_df["regime"] = model_df["state"].map(label_map)


In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))
price_aligned = feat.loc[model_df.index, "close"]
ax1.plot(price_aligned.index, price_aligned.values, color="black", lw=1)
ax1.set_ylabel("Price")

colors = {"Calm/Trending": "#8fd694", "Choppy/Neutral": "#f5d76e", "Stressed/High-Vol": "#e57373"}
for regime_name, color in colors.items():
    mask = model_df["regime"] == regime_name
    ax1.scatter(model_df.index[mask], price_aligned[mask], s=8, color=color, label=regime_name)

ax1.legend(loc="upper left")
ax1.set_title("Detected Regimes over Price")
plt.tight_layout()
plt.show()


## 6. Transition matrix — how persistent/sticky are regimes?

Important for a *few-day* horizon: this tells you the probability of staying in the same
regime tomorrow vs. switching, which is directly usable as a short-horizon forecast.


In [ ]:
trans_df = pd.DataFrame(hmm.transmat_, index=[label_map[i] for i in range(N_STATES)],
                          columns=[label_map[i] for i in range(N_STATES)])
print("Transition probabilities (row = today's regime, col = tomorrow's regime):")
print(trans_df.round(3))

# expected regime duration = 1 / (1 - P(stay))
for st in range(N_STATES):
    p_stay = hmm.transmat_[st, st]
    exp_duration = 1 / (1 - p_stay) if p_stay < 1 else np.inf
    print(f"{label_map[st]}: P(stay)={p_stay:.2f}, expected duration={exp_duration:.1f} days")


## 7. Walk-forward backtest: does the current regime predict the next few days?

This is the key test — fit the HMM **only on data available up to time t** (no lookahead),
then check whether today's inferred regime has predictive value for forward 1/3/5-day
returns and realized volatility. We refit periodically (e.g. every 60 days) rather than
every single day, which is both more realistic and much faster.


In [ ]:
def walk_forward_regimes(feat_df, regime_features, n_states=3,
                          min_train=250, refit_every=60, random_state=42):
    """
    Returns a DataFrame indexed like feat_df with a 'state' column populated
    out-of-sample (each day's state is inferred using only a model trained on
    prior data).
    """
    data = feat_df[regime_features].dropna()
    idx = data.index
    states_oos = pd.Series(index=idx, dtype=float)

    model = None
    scaler_wf = None
    label_map_wf = None

    for i in range(min_train, len(idx)):
        if model is None or (i - min_train) % refit_every == 0:
            train = data.iloc[:i]
            scaler_wf = StandardScaler().fit(train.values)
            Xtr = scaler_wf.transform(train.values)
            model = GaussianHMM(n_components=n_states, covariance_type="full",
                                 n_iter=300, random_state=random_state)
            model.fit(Xtr)
            # relabel by vol rank each refit, using the same feature we used before
            vol_col_idx = regime_features.index("vol_cc_20") if "vol_cc_20" in regime_features else 0
            means = model.means_[:, vol_col_idx]
            order = np.argsort(means)
            names_local = ["Calm/Trending", "Choppy/Neutral", "Stressed/High-Vol"][:n_states]
            label_map_wf = {order[j]: names_local[j] for j in range(n_states)}

        x_today = scaler_wf.transform(data.iloc[[i]].values)
        s = model.predict(x_today)[0]
        states_oos.iloc[i] = s if not isinstance(s, str) else s

    out = feat_df.loc[idx].copy()
    out["state_oos"] = states_oos
    out["regime_oos"] = out["state_oos"].map(lambda s: label_map_wf.get(s, np.nan) if pd.notna(s) else np.nan)
    return out

# NOTE: this loop refits every `refit_every` days, so it is reasonably fast,
# but on very long histories you may want to increase refit_every further.
oos = walk_forward_regimes(feat, regime_features, n_states=N_STATES,
                            min_train=250, refit_every=60)
oos[["close", "state_oos", "regime_oos"]].dropna().tail()


In [ ]:
# Forward returns & forward realized vol, computed from data the regime call did NOT see
for h in [1, 3, 5]:
    oos[f"fwd_ret_{h}d"] = oos["close"].shift(-h) / oos["close"] - 1

oos["fwd_vol_5d"] = oos["logret_1d"].shift(-5).rolling(5).std() * np.sqrt(252)

eval_df = oos.dropna(subset=["regime_oos", "fwd_ret_1d", "fwd_ret_3d", "fwd_ret_5d", "fwd_vol_5d"])

summary = eval_df.groupby("regime_oos").agg(
    n=("fwd_ret_1d", "size"),
    mean_fwd_ret_1d=("fwd_ret_1d", "mean"),
    mean_fwd_ret_3d=("fwd_ret_3d", "mean"),
    mean_fwd_ret_5d=("fwd_ret_5d", "mean"),
    std_fwd_ret_5d=("fwd_ret_5d", "std"),
    mean_fwd_vol_5d=("fwd_vol_5d", "mean"),
)
print("Forward-looking stats by OUT-OF-SAMPLE regime label:")
print(summary.round(4))


### Statistical significance check

A quick one-way ANOVA / Kruskal-Wallis on forward 5-day returns across regimes tells you
whether the regime split is actually informative or just noise.


In [ ]:
from scipy.stats import f_oneway, kruskal

groups = [g["fwd_ret_5d"].values for _, g in eval_df.groupby("regime_oos")]
groups = [g[~np.isnan(g)] for g in groups if len(g) > 5]

if len(groups) >= 2:
    f_stat, p_val = f_oneway(*groups)
    h_stat, p_val_kw = kruskal(*groups)
    print(f"ANOVA on fwd_ret_5d across regimes: F={f_stat:.3f}, p={p_val:.4f}")
    print(f"Kruskal-Wallis (non-parametric): H={h_stat:.3f}, p={p_val_kw:.4f}")
    print("p < 0.05 suggests the regime label carries real predictive information for forward returns.")
else:
    print("Not enough groups/observations for a significance test.")


## 8. Simple tradable rule as a sanity check

E.g.: reduce/avoid exposure when regime == 'Stressed/High-Vol', stay invested otherwise.
Compare vs. buy-and-hold. This is illustrative, not a production strategy (no costs,
slippage, or position sizing beyond a binary on/off).


In [ ]:
strategy_df = oos.dropna(subset=["regime_oos"]).copy()
strategy_df["position"] = np.where(strategy_df["regime_oos"] == "Stressed/High-Vol", 0.0, 1.0)
strategy_df["strategy_ret"] = strategy_df["position"] * strategy_df["ret_1d"].shift(-1)
strategy_df["bh_ret"] = strategy_df["ret_1d"].shift(-1)

strategy_df["strategy_cum"] = (1 + strategy_df["strategy_ret"].fillna(0)).cumprod()
strategy_df["bh_cum"] = (1 + strategy_df["bh_ret"].fillna(0)).cumprod()

plt.figure(figsize=(12, 5))
plt.plot(strategy_df.index, strategy_df["strategy_cum"], label="Regime-filtered strategy")
plt.plot(strategy_df.index, strategy_df["bh_cum"], label="Buy & Hold", alpha=0.7)
plt.legend()
plt.title("Regime-Filtered Exposure vs Buy & Hold (illustrative, no costs)")
plt.ylabel("Cumulative growth of $1")
plt.show()

def perf_stats(returns, freq=252):
    returns = returns.dropna()
    ann_ret = (1 + returns).prod() ** (freq / len(returns)) - 1
    ann_vol = returns.std() * np.sqrt(freq)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else np.nan
    cum = (1 + returns).cumprod()
    max_dd = (cum / cum.cummax() - 1).min()
    return pd.Series({"Ann. Return": ann_ret, "Ann. Vol": ann_vol,
                       "Sharpe": sharpe, "Max Drawdown": max_dd})

print("Strategy:")
print(perf_stats(strategy_df["strategy_ret"]).round(4))
print("\nBuy & Hold:")
print(perf_stats(strategy_df["bh_ret"]).round(4))


## Notes / next steps

- **Swap in your real data** in cell 2 (Option A or B) — the rest of the notebook runs unchanged on any single-ticker OHLCV frame.
- **Costs/slippage** aren't modeled in section 8 — add a transaction cost per regime switch before trusting the Sharpe.
- **`refit_every`** controls the walk-forward speed/realism tradeoff — smaller = more realistic but slower.
- If you later get **intraday or options data**, the natural extensions are: replace `vol_cc_20`/Parkinson/GK with realized-vol-from-intraday (much more efficient estimator), and add implied vol / skew as a forward-looking regime feature — that's the single biggest upgrade available.
- Try `n_components=2` and `n_components=4` and compare the transition matrix persistence and the ANOVA p-value — more states isn't always better, it just needs enough data per state to fit reliably.
